<a href="https://colab.research.google.com/github/elithecreator06/regression-time-series-forecasting/blob/main/logistic-regression-and-gas-price-forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import the libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [ ]:
#Load the breast cancer dataset
data = load_breast_cancer()

#Features (x) and target values (y)
x = data.data
y = data.target

#Split the data into 70% training and 30% testing
x_train, x_test, y_train, y_test = train_test_split(
  x, y,
  test_size = 0.30,
  random_state = 42
)

#sigmoid activation function
#Converts any value inot a pobability between 0 and 1
def sigmoid(z):
  z = np.clip(z, -500, 500)
  return 1 / (1 + np.exp(-z))

In [ ]:
#Logistic Regression class
class LogisticRegressionScratch:

  def __init__(self, learning_rate = 0.1, epochs = 1000):
    self.learning_rate = learning_rate
    self.epochs = epochs

  def fit(self, x, y):

    #Number of samples and features
    samples, features = x.shape

    #Initialise weights and bias
    self.weights = np.zeros(features)
    self.bias = 0

    #Train the model using Gradient Descent
    for i in range(self.epochs):
      #Linear equation
      linear = np.dot(x, self.weights) + self.bias

      #Apply sigmoid function
      predictions = sigmoid(linear)

      #Calculate gradients
      dw = (1 / samples) * np.dot(x.T, (predictions - y))
      db = (1 / samples) * np.sum(predictions - y)

      #Update weights and bias
      self.weights -= self.learning_rate * dw
      self.bias -= self.learning_rate * db

  def predict(self, x):

    #Calculate probabilities
    linear = np.dot(x, self.weights) + self.bias
    predictions = sigmoid(linear)

    #Convert probabilities into class labels
    return [1 if i >= 0.5 else 0 for i in predictions]

In [ ]:
#Create the model
model = LogisticRegressionScratch()

#Train the model
model.fit(x_train, y_train)

#Predict the test data
predictions = model.predict(x_test)

#Calculate the accuracy
accuracy = np.mean(predictions == y_test)

print("Classification Accuracy: ", accuracy)

Classification Accuracy:  0.9649122807017544


PART 2 -- Time Series Forecasting

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 40-Day_Gas_Price_Dataset.csv to 40-Day_Gas_Price_Dataset.csv


In [ ]:
#Load the gas price dataset
gas = pd.read_csv("40-Day_Gas_Price_Dataset.csv")

#Display the first few rows
print(gas.head())

         Date     Price
0  2024-01-01  3.014901
1  2024-01-02  3.005852
2  2024-01-03  3.039431
3  2024-01-04  3.075691
4  2024-01-05  3.032975


In [ ]:
#Convert dates into datetime format
gas["Date"] = pd.to_datetime(gas["Date"])

#Select the gas prices
prices = gas["Price"].values

In [ ]:
#Create sliding window data
window_size = 5

x = []
y = []

for i in range(len(prices) - window_size):
  x.append(prices[i:i + window_size])
  y.append(prices[i + window_size])


x = np.array(x)
y = np.array(y)

In [ ]:
#Initialise the weights and bias
weights = np.zeros(window_size)
bias = 0
learning_rate = 0.001
epochs = 1000

In [ ]:
#Train linear Regression using Gradient Descent
for epoch in range(epochs):
  predictions = np.dot(x, weights) + bias

  error = predictions - y

  #Calculate gradients
  dw = (2 / len(x)) * np.dot(x.T, error)
  db = (2 / len(x)) * np.sum(error)

  #Update parameters
  weights -= learning_rate * dw
  bias -= learning_rate * db

In [ ]:
#Forecast the next seven days
last_window = prices[-window_size:].tolist()

future = []

for i in range(7):
  #Predict the next gas price
  prediction = np.dot(last_window, weights) + bias

  future.append(prediction)

  #Move the sliding window
  last_window.pop(0)
  last_window.append(prediction)

print("Next 7_Day Forecast")

for i, value in enumerate(future, start = 41):
  print(f"Day {i}: {value:.2f}")

Next 7_Day Forecast
Day 41: 3.37
Day 42: 3.38
Day 43: 3.38
Day 44: 3.40
Day 45: 3.41
Day 46: 3.41
Day 47: 3.42
